# Dataset loading from roboflow:

In [1]:
from roboflow import Roboflow
import os

In [3]:

from roboflow import Roboflow
rf = Roboflow(api_key="xChTu2x7E9ULr1Vym0By")
project = rf.workspace("abc-6ewki").project("mealawe_annotation")
version = project.version(1)
dataset = version.download("coco")
                

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to mealawe_annotation-1 in coco:: 100%|██████████| 111/111 [00:00<00:00, 1255.62it/s]


In [16]:
rf = Roboflow(api_key=os.getenv("api_key"))
project = rf.workspace("abc-6ewki").project("mealawe_annotation")
version = project.version(5)
dataset = version.download("tensorflow")
                

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to mealawe_annotation-5 in tensorflow:: 100%|██████████| 1121/1121 [00:00<00:00, 1645.12it/s]


# Dataset modification for yolo format:

In [6]:
import pandas as pd
import cv2
import os


In [ ]:
CLASS_MAP = {
    "Cucumber": 0,
    "Curry": 1,
    "Dal": 2,
    "Onion": 3,
    "Raita": 4,
    "Rice": 5,
    "Roti": 6,
    "Sabzi": 7,
    "Salad": 8,
    "Sweet": 9,
    "Tomato": 10
}


DATASET_PATH = "Dataset"
SPLITS = ["valid", "test"]

In [8]:
def convert_bbox_to_yolo(xmin, ymin, xmax, ymax, img_w, img_h):
    x_center = ((xmin + xmax) / 2) / img_w
    y_center = ((ymin + ymax) / 2) / img_h
    width = (xmax - xmin) / img_w
    height = (ymax - ymin) / img_h
    return x_center, y_center, width, height

In [9]:
def process_split(split):
    print(f"\nProcessing {split}...")

    split_path = os.path.join(DATASET_PATH, split)
    csv_path = os.path.join(split_path, "_annotations.csv")

    df = pd.read_csv(csv_path)

    for filename in df["filename"].unique():
        img_path = os.path.join(split_path, filename)

        img = cv2.imread(img_path)
        h, w, _ = img.shape

        label_file = filename.replace(".jpg", ".txt").replace(".png", ".txt")
        label_path = os.path.join(split_path, label_file)

        rows = df[df["filename"] == filename]

        with open(label_path, "w") as f:
            for _, row in rows.iterrows():

                class_name = row["class"]
                class_id = CLASS_MAP[class_name]

                xmin = row["xmin"]
                ymin = row["ymin"]
                xmax = row["xmax"]
                ymax = row["ymax"]

                x, y, bw, bh = convert_bbox_to_yolo(
                    xmin, ymin, xmax, ymax, w, h
                )

                f.write(f"{class_id} {x:.6f} {y:.6f} {bw:.6f} {bh:.6f}\n")

    print(f"{split} done ✔")



In [15]:
for split in SPLITS:
    process_split(split)

print("\n🎉 All annotations converted to YOLO format!")


Processing valid...
valid done ✔

Processing test...
test done ✔

🎉 All annotations converted to YOLO format!


# Dataset loading in coco format:

In [17]:
import os
import json
import pandas as pd
import cv2

DATASET_PATH = "coco_format_data"
SPLITS = ["train", "valid", "test"]

CATEGORIES = [
    "Cucumber", "Curry", "Dal", "Onion", "Raita",
    "Rice", "Roti", "Sabzi", "Salad", "Sweet", "Tomato"
]

CATEGORY_MAP = {name: i + 1 for i, name in enumerate(CATEGORIES)}  # COCO starts at 1


In [18]:
def process_split(split):
    split_path = os.path.join(DATASET_PATH, split)
    csv_path = os.path.join(split_path, "_annotations.csv")

    df = pd.read_csv(csv_path)

    coco = {
        "images": [],
        "annotations": [],
        "categories": []
    }

    # Categories
    for name, cid in CATEGORY_MAP.items():
        coco["categories"].append({
            "id": cid,
            "name": name,
            "supercategory": "food"
        })

    image_id_map = {}
    ann_id = 1
    img_id = 1

    for filename in df["filename"].unique():
        img_path = os.path.join(split_path, filename)
        img = cv2.imread(img_path)
        h, w, _ = img.shape

        coco["images"].append({
            "id": img_id,
            "file_name": filename,
            "width": w,
            "height": h
        })

        image_id_map[filename] = img_id
        img_id += 1

    for _, row in df.iterrows():
        xmin = row["xmin"]
        ymin = row["ymin"]
        xmax = row["xmax"]
        ymax = row["ymax"]

        width = xmax - xmin
        height = ymax - ymin

        coco["annotations"].append({
            "id": ann_id,
            "image_id": image_id_map[row["filename"]],
            "category_id": CATEGORY_MAP[row["class"]],
            "bbox": [xmin, ymin, width, height],
            "area": width * height,
            "iscrowd": 0
        })

        ann_id += 1

    out_path = os.path.join(split_path, f"{split}.json")
    with open(out_path, "w") as f:
        json.dump(coco, f, indent=4)

    print(f"{split}.json created ✔")

In [19]:
for split in SPLITS:
    process_split(split)

print("\n🎉 COCO conversion complete!")


train.json created ✔
valid.json created ✔
test.json created ✔

🎉 COCO conversion complete!


# COCO Dataset Loader: 

In [1]:
import os
import torch
from torch.utils.data import Dataset
from PIL import Image
from pycocotools.coco import COCO

In [2]:
class UniversalCOCODataset(Dataset):
    def __init__(self, images_dir, annotation_file, transforms=None):
        self.images_dir = images_dir
        self.coco = COCO(annotation_file)
        self.image_ids = list(self.coco.imgs.keys())
        self.transforms = transforms

    def __len__(self):
        return len(self.image_ids)

    def _load_image(self, image_id):
        info = self.coco.loadImgs(image_id)[0]
        path = os.path.join(self.images_dir, info["file_name"])
        return Image.open(path).convert("RGB"), info

    def _load_annotations(self, image_id):
        ann_ids = self.coco.getAnnIds(imgIds=image_id)
        return self.coco.loadAnns(ann_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        image, info = self._load_image(image_id)
        anns = self._load_annotations(image_id)

        if self.transforms:
            image = self.transforms(image)

        return {
            "image": image,
            "annotations": anns,
            "image_id": image_id,
            "image_info": info
        }


In [3]:
def coco_to_torchvision(sample):
    anns = sample["annotations"]

    boxes = []
    labels = []
    areas = []
    iscrowd = []

    for ann in anns:
        x, y, w, h = ann["bbox"]
        boxes.append([x, y, x + w, y + h])
        labels.append(ann["category_id"])
        areas.append(ann["area"])
        iscrowd.append(ann.get("iscrowd", 0))

    target = {
        "boxes": torch.tensor(boxes, dtype=torch.float32),
        "labels": torch.tensor(labels, dtype=torch.int64),
        "image_id": torch.tensor([sample["image_id"]]),
        "area": torch.tensor(areas, dtype=torch.float32),
        "iscrowd": torch.tensor(iscrowd, dtype=torch.int64)
    }

    return sample["image"], target


In [4]:
def coco_to_detr(sample):
    anns = sample["annotations"]
    img_info = sample["image_info"]

    w, h = img_info["width"], img_info["height"]

    boxes = []
    labels = []

    for ann in anns:
        x, y, bw, bh = ann["bbox"]

        cx = (x + bw / 2) / w
        cy = (y + bh / 2) / h
        bw /= w
        bh /= h

        boxes.append([cx, cy, bw, bh])
        labels.append(ann["category_id"])

    target = {
        "boxes": torch.tensor(boxes, dtype=torch.float32),
        "labels": torch.tensor(labels, dtype=torch.int64)
    }

    return sample["image"], target


In [5]:
def coco_to_yolo(sample, category_map):
    """
    category_map: {coco_id: yolo_id}
    """
    anns = sample["annotations"]
    img_info = sample["image_info"]

    w, h = img_info["width"], img_info["height"]

    yolo_labels = []

    for ann in anns:
        x, y, bw, bh = ann["bbox"]

        cx = (x + bw / 2) / w
        cy = (y + bh / 2) / h
        bw /= w
        bh /= h

        cls = category_map[ann["category_id"]]

        yolo_labels.append([cls, cx, cy, bw, bh])

    return sample["image"], torch.tensor(yolo_labels)


In [6]:
def detection_collate_fn(batch):
    images, targets = zip(*batch)
    return list(images), list(targets)
